# copy files to new folder - from txt RK from each element in list 


In [1]:
# # ----------------------------######----------------------------#
# #   _copy_0701_txt_replace_SAFE_GET_df                           #
# # ----------------------------######----------------------------#

# import os
# import shutil
# import codecs
# import pandas as pd
# from tqdm import tqdm


# def _copy_0701_txt_replace_SAFE_GET_df(txt_path,
#                                       path_col='Location',
#                                       sep='\t',
#                                       overwrite_files=True,
#                                       overwrite_txt=True):
#     """
#     Reads a TXT (BOM-aware), copies files listed in df[path_col] into an auto-created
#     dest folder (same dir as txt, named txt basename), replaces df[path_col] with new
#     copied paths, and optionally overwrites the original TXT with updated paths.

#     Adds columns:
#       - copy_status: copied / skipped_exists / missing_source / error
#       - copy_error:  error string if any
#       - src_path:    original path (before replacement)
#       - dest_path:   destination path (after replacement)
#     """

#     if not os.path.isfile(txt_path):
#         raise FileNotFoundError(f"TXT not found: {txt_path}")

#     # ---------- encoding detect (BOM) ----------
#     with open(txt_path, "rb") as f:
#         raw = f.read(4)

#     if raw.startswith(codecs.BOM_UTF16_LE):
#         enc = "utf-16-le"
#         read_enc = "utf-16"   # pandas expects "utf-16" for BOM-driven utf-16
#         write_enc = "utf-16"  # keep BOM on write
#     elif raw.startswith(codecs.BOM_UTF16_BE):
#         enc = "utf-16-be"
#         read_enc = "utf-16"
#         write_enc = "utf-16"
#     elif raw.startswith(codecs.BOM_UTF8):
#         enc = "utf-8-sig"
#         read_enc = "utf-8-sig"
#         write_enc = "utf-8-sig"
#     else:
#         enc = "utf-8"
#         read_enc = "utf-8"
#         write_enc = "utf-8"

#     # ---------- read txt ----------
#     df = pd.read_csv(
#         txt_path,
#         sep=sep,
#         encoding=read_enc,
#         dtype=str,
#         keep_default_na=False,
#         na_filter=False,
#         engine="python"
#     )

#     if path_col not in df.columns:
#         raise KeyError(f"Column '{path_col}' not found in TXT. Columns: {list(df.columns)}")

#     # ---------- dest folder ----------
#     txt_dir = os.path.dirname(txt_path)
#     base = os.path.splitext(os.path.basename(txt_path))[0]
#     dest_dir = os.path.join(txt_dir, base)
#     os.makedirs(dest_dir, exist_ok=True)

#     # ---------- prep cols ----------
#     df["src_path"] = df[path_col].astype(str)
#     df["dest_path"] = ""
#     df["copy_status"] = ""
#     df["copy_error"] = ""

#     # ---------- copy loop (TQDM) ----------
#     for i in tqdm(range(len(df)), desc=f"COPY → {os.path.basename(txt_path)}", unit="file"):
#         src = df.at[i, "src_path"].strip()

#         if (not src) or src.lower() in {"nan", "none"}:
#             df.at[i, "copy_status"] = "missing_source"
#             df.at[i, "copy_error"] = "empty path"
#             continue

#         try:
#             if not os.path.isfile(src):
#                 df.at[i, "copy_status"] = "missing_source"
#                 df.at[i, "copy_error"] = "source not found"
#                 continue

#             fname = os.path.basename(src)
#             dest = os.path.join(dest_dir, fname)

#             # if exists
#             if os.path.isfile(dest) and not overwrite_files:
#                 df.at[i, "dest_path"] = dest
#                 df.at[i, "copy_status"] = "skipped_exists"
#                 df.at[i, "copy_error"] = ""
#                 continue

#             # copy (overwrite if allowed)
#             shutil.copy2(src, dest)

#             df.at[i, "dest_path"] = dest
#             df.at[i, "copy_status"] = "copied"
#             df.at[i, "copy_error"] = ""

#         except Exception as e:
#             df.at[i, "copy_status"] = "error"
#             df.at[i, "copy_error"] = repr(e)

#     # ---------- replace paths ----------
#     df[path_col] = df["dest_path"].where(df["dest_path"].astype(str).str.len() > 0, df[path_col])

#     # ---------- overwrite txt ----------
#     if overwrite_txt:
#         # Write back same columns, preserving your table (plus new cols)
#         df.to_csv(
#             txt_path,
#             sep=sep,
#             index=False,
#             encoding=write_enc
#         )

#     # ---------- quick report ----------
#     vc = df["copy_status"].value_counts(dropna=False).to_dict()
#     print(f"\n✅ TXT DONE: {txt_path}")
#     print(f"DEST FOLDER: {dest_dir}")
#     print("STATUS COUNTS:", vc)

#     return df


In [2]:
# ----------------------------######----------------------------#
#   _copy_0701_txt_replace_SAFE_GET_df                           #
# ----------------------------######----------------------------#

import os
import shutil
import codecs
import csv
import pandas as pd
from tqdm import tqdm


def _copy_0701_txt_replace_SAFE_GET_df(txt_path,
                                      path_col='Location',
                                      sep='\t',
                                      overwrite_files=True,
                                      overwrite_txt=True):
    """
    Reads a TXT (BOM-aware), copies files listed in df[path_col] into an auto-created
    dest folder (same dir as txt, named txt basename), replaces df[path_col] with new
    copied paths, and optionally overwrites the original TXT with updated paths.

    Adds columns:
      - copy_status: copied / skipped_exists / missing_source / error
      - copy_error:  error string if any
      - src_path:    original path (before replacement)
      - dest_path:   destination path (after replacement)
    """

    if not os.path.isfile(txt_path):
        raise FileNotFoundError(f"TXT not found: {txt_path}")

    # ---------- encoding detect (BOM) ----------
    with open(txt_path, "rb") as f:
        raw = f.read(4)

    if raw.startswith(codecs.BOM_UTF16_LE):
        read_enc = "utf-16"     # BOM-driven
        write_enc = "utf-16"    # keep BOM on write
    elif raw.startswith(codecs.BOM_UTF16_BE):
        read_enc = "utf-16"
        write_enc = "utf-16"
    elif raw.startswith(codecs.BOM_UTF8):
        read_enc = "utf-8-sig"
        write_enc = "utf-8-sig"
    else:
        read_enc = "utf-8"
        write_enc = "utf-8"

    # ---------- read txt (HARDENED) ----------
    # NOTE: This disables quote parsing to survive malformed quotes/newlines in fields.
    df = pd.read_csv(
        txt_path,
        sep=sep,
        encoding=read_enc,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        engine="python",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        on_bad_lines="skip",
    )

    if path_col not in df.columns:
        raise KeyError(f"Column '{path_col}' not found in TXT. Columns: {list(df.columns)}")

    # ---------- dest folder ----------
    txt_dir = os.path.dirname(txt_path)
    base = os.path.splitext(os.path.basename(txt_path))[0]
    dest_dir = os.path.join(txt_dir, base)
    os.makedirs(dest_dir, exist_ok=True)

    # ---------- prep cols ----------
    df["src_path"] = df[path_col].astype(str)
    df["dest_path"] = ""
    df["copy_status"] = ""
    df["copy_error"] = ""

    # ---------- copy loop (TQDM) ----------
    for i in tqdm(range(len(df)), desc=f"COPY → {os.path.basename(txt_path)}", unit="file"):
        src = str(df.at[i, "src_path"]).strip()

        if (not src) or src.lower() in {"nan", "none"}:
            df.at[i, "copy_status"] = "missing_source"
            df.at[i, "copy_error"] = "empty path"
            continue

        try:
            if not os.path.isfile(src):
                df.at[i, "copy_status"] = "missing_source"
                df.at[i, "copy_error"] = "source not found"
                continue

            fname = os.path.basename(src)
            dest = os.path.join(dest_dir, fname)

            # if exists
            if os.path.isfile(dest) and not overwrite_files:
                df.at[i, "dest_path"] = dest
                df.at[i, "copy_status"] = "skipped_exists"
                df.at[i, "copy_error"] = ""
                continue

            # copy (overwrite if allowed)
            shutil.copy2(src, dest)

            df.at[i, "dest_path"] = dest
            df.at[i, "copy_status"] = "copied"
            df.at[i, "copy_error"] = ""

        except Exception as e:
            df.at[i, "copy_status"] = "error"
            df.at[i, "copy_error"] = repr(e)

    # ---------- replace paths ----------
    df[path_col] = df["dest_path"].where(df["dest_path"].astype(str).str.len() > 0, df[path_col])

    # ---------- overwrite txt ----------
    if overwrite_txt:
        df.to_csv(
            txt_path,
            sep=sep,
            index=False,
            encoding=write_enc,
            lineterminator="\n",
            quoting=csv.QUOTE_MINIMAL
        )

    # ---------- quick report ----------
    vc = df["copy_status"].value_counts(dropna=False).to_dict()
    print(f"\n✅ TXT DONE: {txt_path}")
    print(f"DEST FOLDER: {dest_dir}")
    print("STATUS COUNTS:", vc)

    return df


#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!


In [5]:
import os

#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!
TXT_FOLDER = '/Users/yerik/Desktop/here'
img_path_cover ='/Users/yerik/Desktop/here /___ashton.png'

In [6]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

import os

txt_paths = [
    os.path.join(TXT_FOLDER, f)
    for f in os.listdir(TXT_FOLDER)
    if f.lower().endswith('.txt') and os.path.isfile(os.path.join(TXT_FOLDER, f))
]

for TXT_PATH in txt_paths:
    print(f'\nRUNNING → {TXT_PATH}')

    df = _copy_0701_txt_replace_SAFE_GET_df(
        txt_path=TXT_PATH,
        path_col='Location',
        sep='\t',
        overwrite_files=True,
        overwrite_txt=True
    )


# rename ALL FILES ACCORDING TO THE CODE WORD AND TXTS TAHT HAVE BEEN UPTADED NEW PATHS 

In [7]:
# ============================================================
# ====== FIX LOCATION PATHS (BOM/CR/QUOTES) + RENAME ==========
# ============================================================

import os
import pandas as pd
from tqdm import tqdm
from datetime import datetime


def _rename_0701_fixpaths_GET_df_from_txt(
    txt_path,
    custom_artist="DJ_Selphi",
    custom_genre="Salsa",
    custom_label="Bachata",
    custom_release_date="",
    custom_purchase_date="",
    sep="\t"
):
    tqdm.pandas()

    # --- BOM detect read (same logic as before, no guessing) ---
    with open(txt_path, "rb") as f:
        head = f.read(4)

    if head.startswith(b"\xff\xfe") or head.startswith(b"\xfe\xff"):
        enc = "utf-16"
    elif head.startswith(b"\xef\xbb\xbf"):
        enc = "utf-8-sig"
    else:
        enc = "utf-16"  # most DJ exports

    df = pd.read_csv(
        txt_path,
        sep=sep,
        encoding=enc,
        engine="python",
        on_bad_lines="skip",
        dtype=str
    )

    df.columns = df.columns.str.strip()

    # ---------- helpers ----------
    def _clean_str(s):
        if s is None:
            return ""
        s = str(s)
        # strip BOM + whitespace + CR
        return s.replace("\ufeff", "").replace("\r", "").strip().strip('"').strip("'")

    def _clean_filename_token(s):
        return (
            _clean_str(s)
            .replace(" ", "_").replace("/", "___").replace(",", "_")
            .replace("(", "").replace(")", "").replace("!", "")
            .replace("&", "and").replace("’", "").replace("'", "")
            .replace("¿", "").replace("¡", "").replace(":", "")
            .replace(";", "").strip()
        )

    def extract_mix(title):
        t = _clean_str(title).lower()
        if ("remix" in t) or ("mix" in t):
            return _clean_filename_token(title)
        return "original"

    def _parse_date_yyyymmdd(val):
        dt = pd.to_datetime(_clean_str(val), errors="coerce")
        return dt.strftime("%Y_%m_%d") if pd.notna(dt) else "NA"

    def _resolve_existing_path(p):
        """
        1) Try as-is
        2) Try expanding ~
        3) If still missing, try locating by filename in same folder as TXT’s auto-folder
           (this helps if Location column is stale but files were copied)
        """
        p = _clean_str(p)
        if not p:
            return None

        p2 = os.path.expanduser(p)
        if os.path.isfile(p2):
            return p2

        # fallback: try find by basename in the auto-folder created from TXT name
        txt_dir = os.path.dirname(txt_path)
        txt_base = os.path.splitext(os.path.basename(txt_path))[0]
        autofolder = os.path.join(txt_dir, txt_base)

        fname = os.path.basename(p2)
        cand = os.path.join(autofolder, fname)
        if os.path.isfile(cand):
            return cand

        return None

    def format_filename(row):
        title = _clean_filename_token(row.get("Track Title", ""))[:25]
        remix = extract_mix(row.get("Track Title", ""))

        artist_val = _clean_filename_token(custom_artist)[:25] if custom_artist else _clean_filename_token(row.get("Artist", ""))[:25]
        genre_val = _clean_filename_token(custom_genre) if custom_genre else _clean_filename_token(row.get("Genre", ""))
        label_val = _clean_filename_token(custom_label) if custom_label else _clean_filename_token(row.get("Label", ""))

        release_date_val = custom_release_date if custom_release_date else _parse_date_yyyymmdd(row.get("Release Date", ""))
        key = _clean_filename_token(row.get("Key", "NA")) or "NA"

        bpm_raw = _clean_str(row.get("BPM", ""))
        try:
            bpm = str(int(round(float(bpm_raw))))
        except Exception:
            bpm = "NA"

        purchase_date_val = custom_purchase_date if custom_purchase_date else _parse_date_yyyymmdd(row.get("Date Added", datetime.today()))

        # extension comes from Location after resolution (safer), but fallback to .mp3
        return title, artist_val, remix, key, bpm, genre_val, label_val, release_date_val, purchase_date_val

    # ---------- main loop ----------
    new_paths = []
    missing_rows = 0

    for i, row in tqdm(df.iterrows(), total=len(df), desc="Renaming files"):
        original_path_raw = row.get("Location", "")
        original_path = _resolve_existing_path(original_path_raw)

        if not original_path:
            missing_rows += 1
            new_paths.append(None)
            continue

        title, artist_val, remix, key, bpm, genre_val, label_val, release_date_val, purchase_date_val = format_filename(row)
        ext = os.path.splitext(original_path)[1] or ".mp3"

        new_filename = (
            f"TRkw_{title}_ARkw_{artist_val}_MXkw_{remix}_KYkw_{key}_"
            f"BPkw_{bpm}_GNkw_{genre_val}_LBkw_{label_val}_RYkw_{release_date_val}_"
            f"PYkw_{purchase_date_val}{ext}"
        )

        if len(new_filename) > 240:
            new_filename = new_filename[:230] + ext

        new_path = os.path.join(os.path.dirname(original_path), new_filename)

        try:
            os.rename(original_path, new_path)
            new_paths.append(new_path)
        except Exception:
            new_paths.append(None)

    df["Renamed_Path"] = new_paths

    if missing_rows:
        print(f"\n⚠️ Missing files for {missing_rows} rows.")
        print("Most likely: Location has stale paths, hidden characters, or copy_status != copied.\n")

    return df


In [8]:
import os

#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!


#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

txt_paths = [
    os.path.join(TXT_FOLDER, f)
    for f in os.listdir(TXT_FOLDER)
    if f.endswith('.txt') and os.path.isfile(os.path.join(TXT_FOLDER, f))  # same-level only
]

for TXT_PATH in txt_paths:
    print(f'RENAME → {TXT_PATH}')

    df = _rename_0701_fixpaths_GET_df_from_txt(
        txt_path=TXT_PATH,
        custom_artist="",
        custom_genre="",
        custom_label=""
    )


# Transform to AIFF


In [9]:
# -----######-----###### CORE IMPORTABLE FUNCTION (All → AIFF 44.1/16/Stereo + Tags + Verify) -----######-----###### #
import os, sys, shutil, subprocess, tempfile
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

# Tagging
from mutagen import File as MutaFile
from mutagen.aiff import AIFF
from mutagen.id3 import (
    ID3, ID3NoHeaderError, ID3BadUnsynchData,
    TIT2, TPE1, TPE2, TALB, TCON, TDRC, TRCK, TPOS, COMM, TBPM, TKEY,
    TPUB, TSRC, TPE3, TCOM, TENC, APIC, CTOC, CHAP
)

# -------------------- helpers (no ASCII banner for sub-fns) -------------------- #
def _safe_get_first(d, key):
    if d is None: return None
    v = d.get(key)
    if v is None: return None
    if isinstance(v, (list, tuple)):
        return v[0] if v else None
    return v

def _as_int_pair(text):
    if not text: return None, None
    s = str(text)
    if '/' in s:
        a,b = s.split('/',1)
        return (a.strip() or None), (b.strip() or None)
    return (s.strip() or None), None

def _ensure_id3(aiff_path):
    a = AIFF(aiff_path)
    if a.tags is None:
        a.add_tags()
    return a

def _copy_id3_frames(src_id3, dst_id3):
    # Copy common frames + chapters/artwork; ignore oddities that fail to serialize.
    for frame in list(src_id3.values()):
        try:
            if isinstance(frame, APIC):
                dst_id3.add(APIC(encoding=frame.encoding, mime=frame.mime, type=frame.type, desc=frame.desc, data=frame.data))
            elif isinstance(frame, COMM):
                dst_id3.add(COMM(encoding=frame.encoding, lang=frame.lang, desc=frame.desc, text=frame.text))
            elif isinstance(frame, (TIT2, TPE1, TPE2, TALB, TCON, TDRC, TRCK, TPOS, TBPM, TKEY, TPUB, TSRC, TPE3, TCOM, TENC)):
                dst_id3.add(type(frame)(encoding=frame.encoding, text=frame.text))
            elif isinstance(frame, (CTOC, CHAP)):
                dst_id3.add(frame)
            else:
                # Pass through for other safe T* frames
                dst_id3.add(frame)
        except Exception:
            # Skip non-serializable frames without killing the run
            pass

def _map_generic_to_id3(vtags, dst_id3, pictures=None):
    # Generic (Vorbis/FLAC/WAV INFO) → ID3
    title   = _safe_get_first(vtags, "title")
    artist  = _safe_get_first(vtags, "artist")
    album   = _safe_get_first(vtags, "album")
    albumartist = _safe_get_first(vtags, "albumartist") or _safe_get_first(vtags, "album artist")
    genre   = _safe_get_first(vtags, "genre")
    date    = _safe_get_first(vtags, "date") or _safe_get_first(vtags, "year")
    comment = _safe_get_first(vtags, "comment") or _safe_get_first(vtags, "description")
    bpm     = _safe_get_first(vtags, "bpm")
    key_    = _safe_get_first(vtags, "initialkey") or _safe_get_first(vtags, "key")
    label   = _safe_get_first(vtags, "label") or _safe_get_first(vtags, "publisher")
    isrc    = _safe_get_first(vtags, "isrc")
    remixer = _safe_get_first(vtags, "remixer")
    composer= _safe_get_first(vtags, "composer")
    encoder = _safe_get_first(vtags, "encoder") or _safe_get_first(vtags, "encodedby") or _safe_get_first(vtags, "encoded_by")
    trk     = _safe_get_first(vtags, "tracknumber")
    dsk     = _safe_get_first(vtags, "discnumber")

    if title:   dst_id3.add(TIT2(encoding=3, text=str(title)))
    if artist:  dst_id3.add(TPE1(encoding=3, text=str(artist)))
    if album:   dst_id3.add(TALB(encoding=3, text=str(album)))
    if albumartist: dst_id3.add(TPE2(encoding=3, text=str(albumartist)))
    if genre:   dst_id3.add(TCON(encoding=3, text=str(genre)))
    if date:    dst_id3.add(TDRC(encoding=3, text=str(date)))
    if comment: dst_id3.add(COMM(encoding=3, lang="eng", desc="", text=str(comment)))
    if bpm:     dst_id3.add(TBPM(encoding=3, text=str(bpm)))
    if key_:    dst_id3.add(TKEY(encoding=3, text=str(key_)))
    if label:   dst_id3.add(TPUB(encoding=3, text=str(label)))
    if isrc:    dst_id3.add(TSRC(encoding=3, text=str(isrc)))
    if remixer: dst_id3.add(TPE3(encoding=3, text=str(remixer)))
    if composer:dst_id3.add(TCOM(encoding=3, text=str(composer)))
    if encoder: dst_id3.add(TENC(encoding=3, text=str(encoder)))

    if trk:
        n, d = _as_int_pair(trk)
        if n or d:
            dst_id3.add(TRCK(encoding=3, text=[f"{n or ''}/{d or ''}".strip('/')]))
    if dsk:
        n, d = _as_int_pair(dsk)
        if n or d:
            dst_id3.add(TPOS(encoding=3, text=[f"{n or ''}/{d or ''}".strip('/')]))

    if pictures:
        for pic in pictures:
            try:
                dst_id3.add(APIC(encoding=3, mime=getattr(pic, "mime", None) or "image/jpeg", type=3, desc=u"", data=getattr(pic, "data", b"")))
            except Exception:
                pass

def _copy_all_tags_to_aiff(src_path, aiff_path):
    """
    After encoding, write a clean ID3 tag set into the AIFF.
    Priority:
      1) If source has ID3 → copy frames
      2) Else, map Vorbis/FLAC/WAV INFO → ID3; copy artwork when possible
    """
    dst_aiff = _ensure_id3(aiff_path)
    dst_id3 = dst_aiff.tags

    src = MutaFile(src_path)
    if src is None:
        dst_aiff.save()
        return

    # Direct ID3 → ID3
    try:
        src_id3 = getattr(src, "tags", None)
        if isinstance(src_id3, ID3) or (src_id3 and any(k.startswith("T") or k in ("APIC","COMM","CTOC","CHAP") for k in src_id3.keys())):
            try:
                _copy_id3_frames(src_id3, dst_id3)
                dst_aiff.save()
                return
            except Exception:
                pass
    except Exception:
        pass

    # Generic mapping (Vorbis/FLAC/WAV INFO, MP4 atoms won't map fully)
    vtags = getattr(src, "tags", {}) or {}
    pictures = []
    try:
        # FLAC: embedded pictures
        if hasattr(src, "pictures") and getattr(src, "pictures", None):
            pictures = src.pictures
        elif hasattr(src, "tags") and "METADATA_BLOCK_PICTURE" in src.tags:
            pictures = []  # base64 case skipped (mutagen handles some variants)
    except Exception:
        pictures = []

    try:
        _map_generic_to_id3(vtags, dst_id3, pictures=pictures)
    except Exception:
        pass

    dst_aiff.save()

def _verify_aiff_ok(aiff_path):
    try:
        t = AIFF(aiff_path)
        _ = t.info.length  # raises if broken
        return True, None
    except Exception as e:
        return False, str(e)

def _exts_casefold(exts):
    # normalize to a case-insensitive set, include upper/lower/Title variants
    s = set()
    for e in exts or []:
        if not e: continue
        ee = e if e.startswith(".") else "."+e
        base = ee.lower()
        s.add(base)
        s.add(base.upper())
        s.add(base.capitalize())
    return s

def _ffmpeg_encode_to_aiff(src_path, dst_path):
    """
    Robust ffmpeg call:
    - force AIFF PCM 16-bit big-endian, 44.1kHz, stereo
    - strip container metadata (we'll write fresh ID3 next)
    - disable video/subs
    - choose first audio stream explicitly
    """
    cmd = [
        "ffmpeg",
        "-hide_banner", "-loglevel", "error",
        "-y",
        "-i", str(src_path),
        "-map", "0:a:0",
        "-vn", "-sn",
        "-ar", "44100",
        "-ac", "2",
        "-c:a", "pcm_s16be",
        "-map_metadata", "-1",
        str(dst_path)
    ]
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return p.returncode == 0

# -----######-----###### CORE IMPORTABLE FUNCTION (per-file) -----######-----###### #
def _aiff_0109_onefile_GET_status_path(
    file_in,
    out_dir=None
):
    """
    Convert a single file → AIFF (44.1kHz / 16-bit / stereo), always re-encode (even AIFF).
    Returns (status, path_out). status in {"ok", "broken", "fail"}.
    """
    src_path = Path(file_in)
    if out_dir is None:
        out_dir = src_path.parent
    else:
        out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)

    dst_path = out_dir / (src_path.stem + ".aiff")

    # Atomic write via temp file to avoid half-written outputs
    with tempfile.TemporaryDirectory() as td:
        tmp_path = Path(td) / (src_path.stem + ".aiff")

        ok = _ffmpeg_encode_to_aiff(src_path, tmp_path)
        if not ok or not tmp_path.exists():
            return "fail", None

        # Write tags/artwork (best-effort; never fatal)
        try:
            _copy_all_tags_to_aiff(src_path, tmp_path)
        except Exception:
            pass

        # Verify playable
        v_ok, v_err = _verify_aiff_ok(tmp_path)
        if not v_ok:
            # Move the broken file out with suffix
            broken_dst = dst_path.with_stem(dst_path.stem + "_BROKEN")
            try:
                broken_dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(tmp_path), str(broken_dst))
            except Exception:
                pass
            return "broken", broken_dst if broken_dst.exists() else dst_path

        # All good: move into place
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(tmp_path), str(dst_path))

    return "ok", dst_path

# -----######-----###### CORE IMPORTABLE FUNCTION (batch) -----######-----###### #
def _aiff_0109_all2aiff_GET_summary(
    root_folder,
    out_root=None,
    audio_extensions=None,
    dry_run="n",
    src_action="keep",          # "keep" | "move" | "trash"
    src_action_move_dir=None,   # required if src_action == "move"
    overwrite="n"               # "y" to overwrite existing AIFF at dst, else skip creating duplicate
):
    """
    Recursively convert *major* audio types to AIFF (CDJ-safe 44.1/16/stereo) with tags & artwork.
    - Always re-encodes, even for AIFF sources (to guarantee target spec)
    - Mirrors folder structure under out_root (if provided)
    - Case-insensitive extension handling
    - macOS junk skipped
    - TQM progress bar + compact summary
    - Post-success actions on sources: keep | move | trash

    Returns:
      summary dict with counts and lists per status.
    """
    root = Path(root_folder)

    # If user doesn't pass, we include a broad set of common types
    if audio_extensions is None:
        audio_extensions = [
            ".flac", ".wav", ".mp3", ".aiff", ".aif",
            ".m4a", ".aac", ".alac", ".ogg", ".oga", ".wv", ".aifc"
        ]
    exts_all = _exts_casefold(audio_extensions)

    # Collect candidates (skip macOS junk)
    all_files = [
        p for p in root.rglob("*")
        if p.is_file()
        and not p.name.startswith("._")
        and p.name != ".DS_Store"
        and (p.suffix in exts_all)
    ]

    # Compute outputs + decide skips
    targets = []
    for src_path in all_files:
        rel = src_path.relative_to(root)
        out_dir = (Path(out_root) / rel.parent) if out_root else src_path.parent
        dst_path = out_dir / (src_path.stem + ".aiff")

        if dry_run.lower().startswith("y"):
            targets.append((src_path, out_dir, dst_path, "todo"))
        else:
            # If overwrite == 'n' and AIFF already exists in destination, skip re-creating file
            if dst_path.exists() and not overwrite.lower().startswith("y"):
                # Still *verify* later if the existing AIFF matches spec? We assume OK for speed.
                # If you want forced re-encode, set overwrite='y'.
                continue
            targets.append((src_path, out_dir, dst_path, "todo"))

    # ----- TQM BAR -----
    pbar = tqdm(total=len(targets), desc="TQM | All → AIFF 44.1/16/stereo", unit="file")

    stats = {
        "ok": 0, "broken": 0, "fail": 0,
        "skipped_existing": 0,
        "total_scanned": len(all_files),
        "total_planned": len(targets),
        "ok_paths": [], "broken_paths": [], "fail_paths": [], "skipped_paths": []
    }

    # If dry-run: just preview names and return summary
    if dry_run.lower().startswith("y"):
        for (src_path, out_dir, dst_path, _) in targets:
            pbar.set_postfix_str(f"DRY-RUN → {src_path.name}")
            pbar.update(1)
            stats["skipped_paths"].append(str(src_path))
        pbar.close()
        return stats

    # Convert
    for (src_path, out_dir, dst_path, _) in targets:
        # If we got here and the destination exists but overwrite == 'n', mark skipped
        if dst_path.exists() and not overwrite.lower().startswith("y"):
            stats["skipped_existing"] += 1
            stats["skipped_paths"].append(str(dst_path))
            pbar.set_postfix_str(f"SKIP (exists): {dst_path.name}")
            pbar.update(1)
            continue

        status, outp = _aiff_0109_onefile_GET_status_path(src_path, out_dir=out_dir)
        if status == "ok":
            stats["ok"] += 1
            stats["ok_paths"].append(str(outp))
            pbar.set_postfix_str(f"OK: {src_path.name}")
            # Post-success action on source
            try:
                if src_action == "trash":
                    # move to user Trash if available; fallback to unlink
                    try:
                        from send2trash import send2trash
                        send2trash(str(src_path))
                    except Exception:
                        src_path.unlink(missing_ok=True)
                elif src_action == "move":
                    if not src_action_move_dir:
                        raise ValueError("src_action_move_dir is required when src_action='move'")
                    dst_dir_move = Path(src_action_move_dir); dst_dir_move.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(src_path), str(dst_dir_move / src_path.name))
                else:
                    pass  # keep
            except Exception:
                # Non-fatal; keep going
                pass

        elif status == "broken":
            stats["broken"] += 1
            stats["broken_paths"].append(str(outp))
            pbar.set_postfix_str(f"BROKEN: {src_path.name}")
        else:
            stats["fail"] += 1
            stats["fail_paths"].append(str(src_path))
            pbar.set_postfix_str(f"FAIL: {src_path.name}")

        pbar.update(1)

    pbar.close()

    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(
        f"[{stamp}] === SUMMARY ===\n"
        f"Total scanned:     {stats['total_scanned']}\n"
        f"Planned to convert:{stats['total_planned']}\n"
        f"Converted OK:      {stats['ok']}\n"
        f"Broken (tagged):   {stats['broken']}\n"
        f"Failed:            {stats['fail']}\n"
        f"Skipped (exists):  {stats['skipped_existing']}\n"
    )
    return stats


In [10]:
# ! CHANGE THESE AS NEEDED
root_folder = TXT_FOLDER


In [11]:
out_root    = None  # or e.g. "/Volumes/HD_back_UP/ALL_MUSIC/_AIFF_OUT"

# Externalized extensions (case-insensitive will be auto-handled)
audio_extensions = [".flac", ".wav", ".mp3", ".aiff", ".aif", ".m4a", ".aac", ".alac", ".ogg", ".oga", ".wv", ".aifc"]

# Action knobs
dry_run     = "n"           # "y" to preview only
overwrite   = "n"           # "y" to force re-encode even if dst exists
src_action  = "keep"        # "keep" | "move" | "trash"
move_dir    = ""  # required if src_action=="move"

summary = _aiff_0109_all2aiff_GET_summary(
    root_folder=root_folder,
    out_root=out_root,
    audio_extensions=audio_extensions,
    dry_run=dry_run,
    src_action=src_action,
    src_action_move_dir=move_dir,
    overwrite=overwrite
)


TQM | All → AIFF 44.1/16/stereo: 100%|████████| 161/161 [04:40<00:00,  1.74s/file, OK: 02. Mutara Speech.flac]

[2026-03-05 21:46:00] === SUMMARY ===
Total scanned:     167
Planned to convert:161
Converted OK:      161
Broken (tagged):   0
Failed:            0
Skipped (exists):  0



# erase originals after checking 

In [10]:
# -----######-----###### CORE IMPORTABLE FUNCTION (Erase everything but AIFF) -----######-----###### #
import os
from pathlib import Path
from tqdm import tqdm

def _cleanup_2208_keepaiff_GET_removed_files(root_folder, dry_run="n"):
    """
    Recursively erase everything but AIFF (.aiff/.aif) files.
    Skips system junk (.DS_Store, ._*).
    
    Inputs:
      root_folder : str/Path → folder to clean
      dry_run     : "y" = preview only, "n" = actually delete
    
    Returns:
      dict summary with counts
    """
    root = Path(root_folder)

    # Collect all files
    all_files = [p for p in root.rglob("*") if p.is_file()]
    # Keep only those NOT AIFF
    targets = [
        p for p in all_files
        if p.suffix.lower() not in (".aiff", ".aif")
        and not p.name.startswith("._")
        and p.name != ".DS_Store"
    ]

    # Progress bar
    pbar = tqdm(total=len(targets), desc="TQM • Cleaning non-AIFF files", unit="file")

    removed, skipped = 0, 0
    for f in targets:
        if dry_run.lower().startswith("y"):
            pbar.set_postfix_str(f"DRY-RUN: would remove {f.name}")
            skipped += 1
        else:
            try:
                f.unlink()
                removed += 1
                pbar.set_postfix_str(f"Removed {f.name}")
            except Exception as e:
                skipped += 1
                pbar.set_postfix_str(f"⚠️ Skip {f.name}: {e}")
        pbar.update(1)

    pbar.close()

    summary = {
        "total_files": len(all_files),
        "removed": removed,
        "skipped": skipped,
        "kept_aiff": len(all_files) - len(targets),
    }

    print(
        f"Done cleanup.\n"
        f" • Total files scanned: {summary['total_files']}\n"
        f" • Removed: {summary['removed']}\n"
        f" • Skipped (errors/dry-run): {summary['skipped']}\n"
        f" • Kept AIFF: {summary['kept_aiff']}\n"
    )
    return summary


In [11]:
_cleanup_2208_keepaiff_GET_removed_files(root_folder, dry_run="n")


TQM • Cleaning non-AIFF files: 100%|█| 228/228 [00:00<00:00, 1753.54file/s, Removed TRkw_Scan_7_-_Black_Highwa

Done cleanup.
 • Total files scanned: 456
 • Removed: 228
 • Skipped (errors/dry-run): 0
 • Kept AIFF: 228



{'total_files': 456, 'removed': 228, 'skipped': 0, 'kept_aiff': 228}

# EMBED IMAGE COVER 

In [12]:
# ----------------------------######----------------------------#
#   _cover_0801_aiffRECUR_GET_rewrite_verify_atomicreplace     #
# ----------------------------######----------------------------#

import os
import tempfile
from tqdm import tqdm
from mutagen.aiff import AIFF
from mutagen.id3 import APIC


def _cover_0801_aiffRECUR_GET_rewrite_verify_atomicreplace(root_folder, img_path):
    """
    AIFF ONLY (recursive), Beatport-style rewrite:
    - Walk ALL subfolders
    - For every .aif/.aiff:
        1) copy to temp (same folder)
        2) write cover art as ID3v2.3 APIC (prefers JPEG bytes)
        3) re-open temp and VERIFY APIC exists
        4) ONLY THEN delete original and replace (same filename)

    Returns: errors list of (path, error)
    """

    if not os.path.isdir(root_folder):
        raise ValueError("root_folder not found")

    if not os.path.isfile(img_path):
        raise ValueError("img_path not found")

    # ---- load image bytes, prefer JPEG (Beatport/CDJ friendly) ----
    cover_mime = "image/jpeg"
    cover_bytes = None

    try:
        from PIL import Image
        from io import BytesIO

        im = Image.open(img_path).convert("RGB")
        buf = BytesIO()
        im.save(buf, format="JPEG", quality=92, optimize=True)
        cover_bytes = buf.getvalue()

    except Exception:
        # fallback: raw file bytes (might be PNG)
        with open(img_path, "rb") as f:
            cover_bytes = f.read()
        # try to detect png signature
        if cover_bytes[:8] == b"\x89PNG\r\n\x1a\n":
            cover_mime = "image/png"
        else:
            cover_mime = "image/jpeg"

    # ---- gather AIFF files recursively ----
    aiff_files = []
    for dirpath, dirnames, filenames in os.walk(root_folder):
        dirnames[:] = [d for d in dirnames if d not in ("__MACOSX",) and not d.startswith(".")]
        for fn in filenames:
            if fn.startswith("._") or fn.startswith(".DS"):
                continue
            if fn.lower().endswith((".aif", ".aiff")):
                aiff_files.append(os.path.join(dirpath, fn))

    errors = []

    for src_path in tqdm(aiff_files, desc="Rewrite AIFF cover (verify then replace)", unit="file"):
        tmp_path = None
        try:
            src_dir = os.path.dirname(src_path)
            suffix = os.path.splitext(src_path)[1]

            fd, tmp_path = tempfile.mkstemp(prefix=".__tmp__", suffix=suffix, dir=src_dir)
            os.close(fd)

            # 1) byte copy into temp (preserve full file content)
            with open(src_path, "rb") as r, open(tmp_path, "wb") as w:
                w.write(r.read())

            # 2) write ID3 APIC into AIFF temp
            a = AIFF(tmp_path)
            if a.tags is None:
                a.add_tags()

            a.tags.delall("APIC")
            a.tags.add(
                APIC(
                    encoding=3,
                    mime=cover_mime,
                    type=3,
                    desc="Cover",
                    data=cover_bytes
                )
            )

            # force v2.3 (most compatible with Rekordbox/CDJs)
            a.save(v2_version=3)

            # 3) VERIFY: reopen and confirm APIC exists + reasonable size
            b = AIFF(tmp_path)
            if b.tags is None:
                raise RuntimeError("VERIFY FAIL: tags missing after save")

            apics = b.tags.getall("APIC")
            if not apics:
                raise RuntimeError("VERIFY FAIL: APIC not present after save")

            # ensure it’s not an empty/invalid picture
            pic_len = len(apics[0].data) if apics[0].data else 0
            if pic_len < 10_000:  # too small = likely broken
                raise RuntimeError(f"VERIFY FAIL: APIC too small ({pic_len} bytes)")

            # 4) atomic replace ONLY AFTER verification
            os.remove(src_path)
            os.replace(tmp_path, src_path)

        except Exception as e:
            errors.append((src_path, str(e)))
            try:
                if tmp_path and os.path.exists(tmp_path):
                    os.remove(tmp_path)
            except Exception:
                pass

    return errors


In [13]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#

root_folder = TXT_FOLDER
img_path    = img_path_cover  # can be png or jpg

errors = _cover_0801_aiffRECUR_GET_rewrite_verify_atomicreplace(root_folder, img_path)

if errors:
    print("\n⚠️ FAILED FILES (first 30):")
    for p, err in errors[:30]:
        print(" -", p, "->", err)
    if len(errors) > 30:
        print(f"...and {len(errors)-30} more")
else:
    print("\n✅ DONE: All AIFF rewritten with embedded cover (verified).")


Rewrite AIFF cover (verify then replace): 100%|███████████████████████████| 228/228 [00:12<00:00, 17.56file/s]


✅ DONE: All AIFF rewritten with embedded cover (verified).


# normalize 

In [14]:
# ----------------------------######----------------------------#
#   _norm_0801_aiff_GET_inplace_noclip                           #
# ----------------------------######----------------------------#

import os
import re
import math
import shutil
import subprocess
import pandas as pd
from tqdm import tqdm

def _norm_0801_aiff_GET_inplace_noclip(folder_path,
                                      target_peak_db=-1.0,
                                      recursive=True,
                                      dry_run=True,
                                      keep_temp=False,
                                      ffmpeg_path="ffmpeg",
                                      ffprobe_path="ffprobe"):
    """
    Normalize ALL .aiff/.aif in a folder (and subfolders if recursive=True) IN PLACE.
    - Peak-based normalization (NOT LUFS).
    - Hard safety against clipping via alimiter at target_peak_db.
    - Preserves ALL tags + cover art by copying ID3 frames with mutagen.
    - Does NOT rename files. Does NOT duplicate final outputs.

    Safety behavior:
    - Writes a temp file next to each original: .__TMP__NORM__{filename}
    - Verifies the resulting max_volume <= target_peak_db (+ tolerance).
    - Only then replaces original atomically (os.replace).
    - If anything fails, original stays untouched.

    Requirements:
    - ffmpeg + ffprobe installed
    - mutagen installed (pip install mutagen)
    """

    try:
        from mutagen.aiff import AIFF
        from mutagen.id3 import ID3
    except Exception as e:
        raise RuntimeError("mutagen is required. Install with: pip install mutagen") from e

    def _run(cmd):
        return subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    def _ffprobe_audio(in_path):
        cmd = [
            ffprobe_path, "-v", "error",
            "-select_streams", "a:0",
            "-show_entries", "stream=sample_rate,channels,sample_fmt,bits_per_sample,codec_name",
            "-of", "default=noprint_wrappers=1",
            in_path
        ]
        r = _run(cmd)
        if r.returncode != 0:
            raise RuntimeError(r.stderr.strip() or "ffprobe failed")
        info = {}
        for line in r.stdout.splitlines():
            if "=" in line:
                k, v = line.strip().split("=", 1)
                info[k] = v
        return info

    def _parse_max_volume_db(ffmpeg_stderr_text):
        m = re.search(r"max_volume:\s*([-\d\.]+)\s*dB", ffmpeg_stderr_text)
        if m:
            return float(m.group(1))
        m2 = re.search(r"max_volume:\s*(-inf)\s*dB", ffmpeg_stderr_text)
        if m2:
            return float("-inf")
        return None

    def _volumedetect_max_db(in_path):
        cmd = [
            ffmpeg_path, "-hide_banner", "-nostdin", "-v", "info",
            "-i", in_path,
            "-af", "volumedetect",
            "-f", "null", "-"
        ]
        r = _run(cmd)
        if r.returncode != 0:
            raise RuntimeError(r.stderr.strip() or "ffmpeg volumedetect failed")
        max_db = _parse_max_volume_db(r.stderr)
        if max_db is None:
            raise RuntimeError("Could not parse max_volume from ffmpeg volumedetect output")
        return max_db

    def _aiff_codec_from_probe(probe):
        bits = probe.get("bits_per_sample", "")
        sample_fmt = probe.get("sample_fmt", "")
        if bits == "24":
            return "pcm_s24be"
        if bits == "32":
            if "flt" in sample_fmt:
                return "pcm_f32be"
            return "pcm_s32be"
        if bits == "16":
            return "pcm_s16be"
        if "s32" in sample_fmt:
            return "pcm_s32be"
        if "s24" in sample_fmt:
            return "pcm_s24be"
        if "flt" in sample_fmt:
            return "pcm_f32be"
        return "pcm_s16be"

    def _copy_id3_all_frames(src_aiff_path, dst_aiff_path):
        src = AIFF(src_aiff_path)
        if not getattr(src, "tags", None):
            return

        dst = AIFF(dst_aiff_path)
        if not getattr(dst, "tags", None):
            dst.add_tags()

        try:
            dst.tags = ID3()
        except Exception:
            for k in list(dst.tags.keys()):
                del dst.tags[k]

        for frame in src.tags.values():
            dst.tags.add(frame)

        dst.save()

    def _linear_from_db(db):
        return 10 ** (db / 20.0)

    def _iter_aiff_files(root, recursive_flag=True):
        if recursive_flag:
            for dirpath, dirnames, filenames in os.walk(root):
                # skip hidden dirs
                dirnames[:] = [d for d in dirnames if not d.startswith(".")]
                for fn in filenames:
                    if fn.startswith("._") or fn.startswith(".DS"):
                        continue
                    ext = os.path.splitext(fn)[1].lower()
                    if ext in [".aiff", ".aif"]:
                        yield os.path.join(dirpath, fn)
        else:
            for fn in os.listdir(root):
                if fn.startswith("._") or fn.startswith(".DS"):
                    continue
                ext = os.path.splitext(fn)[1].lower()
                if ext in [".aiff", ".aif"]:
                    yield os.path.join(root, fn)

    folder_path = os.path.abspath(os.path.expanduser(str(folder_path)))
    if not os.path.isdir(folder_path):
        raise ValueError("folder_path is not a folder: " + folder_path)

    aiff_files = list(_iter_aiff_files(folder_path, recursive))
    if len(aiff_files) == 0:
        return pd.DataFrame([{
            "Path": folder_path,
            "status": "no_aiff_found",
            "max_db_before": None,
            "max_db_after": None,
            "gain_db": None,
            "error": None
        }])

    target_linear = _linear_from_db(target_peak_db)
    tol_db = 0.15

    rows = []

    for in_path in tqdm(aiff_files, desc="Normalizing AIFF (recursive, no-clip)", unit="file"):
        out_tmp = None
        try:
            probe = _ffprobe_audio(in_path)
            codec = _aiff_codec_from_probe(probe)

            max_db_before = _volumedetect_max_db(in_path)

            if max_db_before == float("-inf"):
                gain_db = 0.0
            else:
                gain_db = float(target_peak_db) - float(max_db_before)

            base = os.path.basename(in_path)
            out_tmp = os.path.join(os.path.dirname(in_path), f".__TMP__NORM__{base}")

            if abs(gain_db) < 1e-6:
                af = f"alimiter=limit={target_linear}:level=enabled"
            else:
                af = f"volume={gain_db}dB,alimiter=limit={target_linear}:level=enabled"

            ff_cmd = [
                ffmpeg_path, "-hide_banner", "-nostdin", "-y",
                "-i", in_path,
                "-map", "0:a:0",
                "-vn",
                "-c:a", codec,
                "-af", af,
                "-map_metadata", "-1",
                out_tmp
            ]

            if dry_run:
                rows.append({
                    "Path": in_path,
                    "status": "dry_run",
                    "max_db_before": max_db_before if max_db_before != float("-inf") else None,
                    "max_db_after": None,
                    "gain_db": gain_db,
                    "error": None
                })
                continue

            r = _run(ff_cmd)
            if r.returncode != 0:
                raise RuntimeError(r.stderr.strip() or "ffmpeg normalize failed")

            _copy_id3_all_frames(in_path, out_tmp)

            max_db_after = _volumedetect_max_db(out_tmp)
            if max_db_after != float("-inf"):
                if max_db_after > (target_peak_db + tol_db):
                    raise RuntimeError(
                        f"Safety check failed: max_db_after={max_db_after} dB > target={target_peak_db} dB"
                    )

            os.replace(out_tmp, in_path)
            out_tmp = None

            rows.append({
                "Path": in_path,
                "status": "normalized_ok",
                "max_db_before": max_db_before if max_db_before != float("-inf") else None,
                "max_db_after": max_db_after if max_db_after != float("-inf") else None,
                "gain_db": gain_db,
                "error": None
            })

        except Exception as e:
            rows.append({
                "Path": in_path,
                "status": "failed_no_change",
                "max_db_before": None,
                "max_db_after": None,
                "gain_db": None,
                "error": str(e)
            })
            if out_tmp and os.path.exists(out_tmp) and not keep_temp:
                try:
                    os.remove(out_tmp)
                except Exception:
                    pass

    return pd.DataFrame(rows)


In [15]:

folder_path =TXT_FOLDER

df_norm = _norm_0801_aiff_GET_inplace_noclip(
    folder_path=folder_path,
    target_peak_db=-1.0,
    recursive=True,
    dry_run=False
)

print(df_norm)
print("\nSTATUS COUNTS:\n", df_norm["status"].value_counts(dropna=False))

Normalizing AIFF (recursive, no-clip): 100%|██████████████████████████████| 228/228 [01:34<00:00,  2.42file/s]

                                                  Path            status  \
0    /Users/yerik/Music/try_new mp3/_4_USA_NIGGD/US...  failed_no_change   
1    /Users/yerik/Music/try_new mp3/_4_USA_NIGGD/US...  failed_no_change   
2    /Users/yerik/Music/try_new mp3/_4_USA_NIGGD/US...  failed_no_change   
3    /Users/yerik/Music/try_new mp3/_4_USA_NIGGD/US...  failed_no_change   
4    /Users/yerik/Music/try_new mp3/_4_USA_NIGGD/US...  failed_no_change   
..                                                 ...               ...   
223  /Users/yerik/Music/try_new mp3/_4_USA_NIGGD/US...  failed_no_change   
224  /Users/yerik/Music/try_new mp3/_4_USA_NIGGD/US...  failed_no_change   
225  /Users/yerik/Music/try_new mp3/_4_USA_NIGGD/US...  failed_no_change   
226  /Users/yerik/Music/try_new mp3/_4_USA_NIGGD/US...  failed_no_change   
227  /Users/yerik/Music/try_new mp3/_4_USA_NIGGD/US...  failed_no_change   

    max_db_before max_db_after gain_db  \
0            None         None    None   
1  

# DONE 